# Maximal covering location problem

Choose at most p candidate facilities to maximize weighted demand reached within a service threshold. This is not p-median: the objective is covered demand, not total assignment cost.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'implementations/python/maximal-covering'))
import numpy as np
import matplotlib.pyplot as plt
from maximal_covering import solve_maximal_covering


## 1. A hypothetical city

Rows are demand locations; columns are candidate sites. Costs are travel-time minutes and weights are illustrative population counts, not observed data.

In [ ]:
costs = np.array([[4,18,22,12,16],[7,14,19,9,13],[16,5,11,17,12],[20,8,9,15,7],[12,16,4,10,18],[17,11,8,13,5]])
weights = np.array([1200,800,950,700,1100,650])
threshold, p = 10, 2
sites, covered_weight, covered = solve_maximal_covering(costs, p, threshold, demand_weights=weights)
print('Selected sites:', sites)
print('Covered population:', covered_weight, 'of', weights.sum())
print('Covered share:', covered_weight / weights.sum())
print('Covered flags:', covered)


## 2. Inspect eligible coverage

Each cell indicates whether a candidate can reach a demand point within the threshold. A demand point counts once even when multiple sites cover it.

In [ ]:
eligible = costs <= threshold
fig, ax = plt.subplots(figsize=(8,4))
ax.imshow(eligible, aspect='auto', cmap='Greys', vmin=0, vmax=1)
ax.set_xticks(range(costs.shape[1]), [f'Site {j}' for j in range(costs.shape[1])])
ax.set_yticks(range(costs.shape[0]), [f'Demand {i}' for i in range(costs.shape[0])])
ax.set_title(f'Eligible coverage within {threshold} minutes')
ax.set_xlabel('Candidate facilities'); ax.set_ylabel('Demand locations')
for i in range(costs.shape[0]):
    for j in range(costs.shape[1]): ax.text(j,i,'✓' if eligible[i,j] else '·',ha='center',va='center')
plt.tight_layout(); plt.show()


## 3. Sensitivity to budget and service standard

The service threshold is a planning choice and should be justified independently of the resulting coverage.

In [ ]:
for service_minutes in (8, 10, 15):
    print(f'\\nThreshold: {service_minutes} minutes')
    for budget in (1, 2, 3):
        chosen, value, flags = solve_maximal_covering(costs, budget, service_minutes, demand_weights=weights)
        print(f'p={budget}: sites={chosen}, covered={value:.0f}, share={value/weights.sum():.1%}')


## 4. Model limitations

This exact enumerator is for small instances. It has no capacity, congestion, stochastic reliability, outages, equity objective or multi-period demand. It accepts a supplied cost matrix and does not build the travel network. Compare with p-median, which minimizes weighted assignment cost rather than maximizing demand within a threshold.